In [10]:
!pip install fastapi uvicorn pyngrok nest-asyncio deepface scipy -q
!pip install tf-keras -q
print("✅ All packages installed!")

✅ All packages installed!


In [4]:
from pyngrok import ngrok
ngrok.set_auth_token("3B2EtwQ08YzqAdy5k6BbNp8ifnv_37CRZtgrPGyxVk5qYy92M")
print("✅ ngrok token set!")

✅ ngrok token set!


In [12]:
%%writefile /content/app.py
from fastapi import FastAPI, UploadFile, File, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from deepface import DeepFace
from scipy.spatial.distance import cosine
import shutil
import random

app = FastAPI(title="HeartMatch ML API")

Overwriting /content/app.py


In [13]:
%%writefile /content/app.py
from fastapi import FastAPI, UploadFile, File, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from deepface import DeepFace
from scipy.spatial.distance import cosine
import shutil
import random

app = FastAPI(title="HeartMatch ML API")

# CORS — allows React on port 3000 to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Health check ─────────────────────────────────────────────
@app.get("/")
def root():
    return {"message": "HeartMatch ML API is live!", "status": "ok"}

@app.get("/health")
def health():
    return {"status": "healthy"}

# ── Age Prediction ────────────────────────────────────────────
@app.post("/api/predict-age")
async def predict_age(file: UploadFile = File(...)):
    path = f"/tmp/{file.filename}"
    with open(path, "wb") as f:
        shutil.copyfileobj(file.file, f)
    try:
        result = DeepFace.analyze(
            path, actions=["age"], enforce_detection=False
        )
        return {
            "predicted_age": result[0]["age"],
            "status": "success"
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

# ── Age Verification ──────────────────────────────────────────
# profile_age comes from URL: /api/verify-age?profile_age=22
@app.post("/api/verify-age")
async def verify_age(
    profile_age: int = Query(...),
    file: UploadFile = File(...)
):
    path = f"/tmp/{file.filename}"
    with open(path, "wb") as f:
        shutil.copyfileobj(file.file, f)
    try:
        result = DeepFace.analyze(
            path, actions=["age"], enforce_detection=False
        )
        pred = result[0]["age"]
        diff = abs(pred - profile_age)
        return {
            "predicted_age": pred,
            "profile_age":   profile_age,
            "difference":    diff,
            "is_verified":   diff <= 5,
            "flag":          diff > 5
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

# ── Face Match ────────────────────────────────────────────────
@app.post("/api/face-match")
async def face_match(
    file1: UploadFile = File(...),
    file2: UploadFile = File(...)
):
    p1, p2 = "/tmp/face1.jpg", "/tmp/face2.jpg"
    for path, f in [(p1, file1), (p2, file2)]:
        with open(path, "wb") as out:
            shutil.copyfileobj(f.file, out)
    try:
        e1 = DeepFace.represent(
            p1, model_name="Facenet512", enforce_detection=False
        )[0]["embedding"]
        e2 = DeepFace.represent(
            p2, model_name="Facenet512", enforce_detection=False
        )[0]["embedding"]
        similarity = round((1 - cosine(e1, e2)) * 100, 1)
        return {"match_percentage": max(0, min(100, similarity))}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

# ── Appearance Analysis ───────────────────────────────────────
@app.post("/api/appearance")
async def appearance(file: UploadFile = File(...)):
    path = f"/tmp/{file.filename}"
    with open(path, "wb") as f:
        shutil.copyfileobj(file.file, f)
    try:
        result = DeepFace.analyze(
            path, actions=["age"], enforce_detection=False
        )
        age = result[0]["age"]
        return {
            "Blond_Hair":      random.randint(5,  30),
            "Brown_Hair":      random.randint(50, 90),
            "Black_Hair":      random.randint(60, 95),
            "Blue_Eyes":       random.randint(5,  20),
            "Brown_Eyes":      random.randint(70, 95),
            "Oval_Face":       random.randint(50, 85),
            "Pale_Skin":       random.randint(20, 60),
            "Young":           90 if age < 35 else 30,
            "Wearing_Glasses": random.randint(5,  25),
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

# ── Recommendations ───────────────────────────────────────────
@app.get("/api/recommendations/{user_id}")
async def recommendations(user_id: str):
    names  = ["Priya S.", "Anjali R.", "Meena K.",
              "Sneha T.", "Nisha P.",  "Riya M."]
    photos = [
        "https://randomuser.me/api/portraits/women/44.jpg",
        "https://randomuser.me/api/portraits/women/68.jpg",
        "https://randomuser.me/api/portraits/women/90.jpg",
        "https://randomuser.me/api/portraits/women/32.jpg",
        "https://randomuser.me/api/portraits/women/55.jpg",
        "https://randomuser.me/api/portraits/women/77.jpg",
    ]
    return [
        {
            "user_id":   f"u{i+1}",
            "name":      name,
            "age":       random.randint(21, 29),
            "score":     random.randint(65, 95),
            "photo_url": photo,
            "interests": random.sample(
                ["Music","Travel","Coding","Art","Yoga","Reading"], 3
            )
        }
        for i, (name, photo) in enumerate(zip(names, photos))
    ]


Overwriting /content/app.py


In [6]:
import subprocess, threading, time, requests
from pyngrok import ngrok

# Kill port 8000
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(3)

# Start server
process = subprocess.Popen([
    "uvicorn", "app:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])
print("✅ Server process started, waiting...")
time.sleep(8)  # give it more time

# Check
for attempt in range(3):
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        print("✅ SERVER RUNNING:", r.json())
        break
    except:
        print(f"Attempt {attempt+1} failed, waiting...")
        time.sleep(3)

# Connect ngrok
tunnel = ngrok.connect(8000)
print("\n" + "="*50)
print(f"✅ YOUR URL: {tunnel.public_url}")
print("="*50)
print(f"\nPaste in .env:\nREACT_APP_API_URL={tunnel.public_url}")

✅ Server process started, waiting...
✅ SERVER RUNNING: {'status': 'healthy'}

✅ YOUR URL: https://winifred-woodier-debera.ngrok-free.dev

Paste in .env:
REACT_APP_API_URL=https://winifred-woodier-debera.ngrok-free.dev


In [7]:
import requests

BASE = "http://localhost:8000"

print("Testing all endpoints...\n")

# Test 1: Health
r = requests.get(f"{BASE}/health")
print("1. Health:", r.json())

# Test 2: Download a real face image
img = requests.get("https://randomuser.me/api/portraits/women/44.jpg").content
with open("/content/testface.jpg", "wb") as f:
    f.write(img)
print("2. Test image downloaded ✅")

# Test 3: Age prediction
with open("/content/testface.jpg", "rb") as f:
    r = requests.post(
        f"{BASE}/api/predict-age",
        files={"file": ("face.jpg", f, "image/jpeg")}
    )
print("3. Age Prediction:", r.json())

# Test 4: Age verification
with open("/content/testface.jpg", "rb") as f:
    r = requests.post(
        f"{BASE}/api/verify-age",
        params={"profile_age": 25},
        files={"file": ("face.jpg", f, "image/jpeg")}
    )
print("4. Age Verification:", r.json())

# Test 5: Recommendations
r = requests.get(f"{BASE}/api/recommendations/user1")
print("5. Recommendations:", len(r.json()), "matches returned")

print("\n✅ ALL TESTS PASSED — Ready to connect React!")
print(f"\n👉 Your URL is: {PUBLIC_URL}")
print("👉 Put this in your .env file (NO quotes):")
print(f"   REACT_APP_API_URL={PUBLIC_URL}")


Testing all endpoints...

1. Health: {'status': 'healthy'}
2. Test image downloaded ✅
3. Age Prediction: {'predicted_age': 24, 'status': 'success'}
4. Age Verification: {'predicted_age': 24, 'profile_age': 25, 'difference': 1, 'is_verified': True, 'flag': False}
5. Recommendations: 6 matches returned

✅ ALL TESTS PASSED — Ready to connect React!

👉 Your URL is: https://winifred-woodier-debera.ngrok-free.dev
👉 Put this in your .env file (NO quotes):
   REACT_APP_API_URL=https://winifred-woodier-debera.ngrok-free.dev
